# Chapter 05 — Writing GPU Kernels with Triton

**Goal:** Understand why kernel fusion matters, learn the GPU memory
hierarchy, and write real Triton kernels that eliminate HBM roundtrips.

**Reference model:** LLaMA-3.2-1B  
d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers

**Hardware:** A100-80GB — 1,774 GB/s bandwidth, 312 TFLOPS FP16 (237 TFLOPS effective), ~134 FLOPS/byte ridge point

## 1 — The Problem: HBM Roundtrips

PyTorch executes operations one at a time. Each op reads from HBM
and writes back to HBM:

```
x = input                          # in HBM
x_norm = rms_norm(x)               # read from HBM → compute → write to HBM
x_act  = silu(x_norm)              # read from HBM → compute → write to HBM
output = linear(x_act, W)          # read from HBM → compute → write to HBM
```

Each intermediate result (`x_norm`, `x_act`) makes a **roundtrip**
through HBM even though the next op will immediately read it back.

These ops are elementwise — very low arithmetic intensity.
The GPU spends most of its time waiting for HBM, not computing.

In [ ]:
# ── Wasted Bandwidth Calculator ──────────────────────────────────────

# Tensor shape: (batch=512, d_model=2048), FP16
batch = 512
d_model = 2048
dtype_bytes = 2  # FP16
tensor_bytes = batch * d_model * dtype_bytes

# ── Unfused: RMSNorm + SiLU ──
# YOUR CODE
# RMSNorm unfused:
#   Step 1: read x, compute variance → write variance       = read tensor_bytes + write (batch * dtype_bytes)
#   Step 2: read x, read variance, normalize → write x_norm = read tensor_bytes + read (batch*dtype) + write tensor_bytes
#   Total reads/writes ≈ 2 * read(tensor) + 2 * write(tensor) = 4 * tensor_bytes  (approximately)
#
# SiLU unfused:
#   read x_norm → compute silu → write x_act = 2 * tensor_bytes
#
# Total unfused HBM traffic ≈ 6 * tensor_bytes
#
# Fused: read x once → RMSNorm → SiLU → write output once = 2 * tensor_bytes
#
# Hint: wasted_bytes = unfused_bytes - fused_bytes

unfused_bytes = 6 * tensor_bytes   # approximate
fused_bytes   = 2 * tensor_bytes   # one read, one write
wasted_bytes  = unfused_bytes - fused_bytes

a100_bw = 1774e9  # bytes/sec
wasted_time_us = wasted_bytes / a100_bw * 1e6

print(f"Tensor size: ({batch}, {d_model}) FP16 = {tensor_bytes / 1e6:.2f} MB")
print(f"Unfused HBM traffic: {unfused_bytes / 1e6:.2f} MB")
print(f"Fused HBM traffic:   {fused_bytes / 1e6:.2f} MB")
print(f"Wasted bandwidth:    {wasted_bytes / 1e6:.2f} MB ({wasted_time_us:.1f} μs on A100)")
print(f"Bandwidth reduction: {unfused_bytes / fused_bytes:.0f}×")
print()
print("For a full forward pass with 16 layers, each with multiple such ops,")
print("the wasted bandwidth adds up to milliseconds — significant at decode time.")

## 2 — GPU Memory Hierarchy

| Level | Size (A100) | Latency | Bandwidth |
|-------|------------|---------|----------|
| **Registers** | ~256 KB/SM | 1 cycle | Fastest |
| **Shared Memory (SRAM)** | ~192 KB/SM (configurable) | ~20 cycles | ~19 TB/s aggregate |
| **L2 Cache** | 40 MB | ~200 cycles | ~5 TB/s |
| **HBM (Global Memory)** | 80 GB | 300-400 cycles | 1,774 GB/s |

The key insight: **SRAM is 200× faster than HBM but 400,000× smaller.**

Kernel fusion works by keeping intermediates in SRAM/registers
instead of writing them to HBM between operations.

In [ ]:
# ── Total SRAM across all SMs ────────────────────────────────────────

sram_per_sm_kb = 192   # KB (configurable up to 192 on A100)
num_sms = 108          # A100 has 108 SMs

# YOUR CODE
# Hint: total_sram_mb = sram_per_sm_kb * num_sms / 1024
total_sram_mb = sram_per_sm_kb * num_sms / 1024

print(f"SRAM per SM:      {sram_per_sm_kb} KB")
print(f"Number of SMs:    {num_sms}")
print(f"Total SRAM:       {total_sram_mb:.1f} MB")
print()

# Why model can't fit in SRAM
llama_1b_bytes = 1.24e9 * 2  # 1.24B params × 2 bytes (FP16)
print(f"LLaMA-3.2-1B model size:   {llama_1b_bytes / 1e6:.0f} MB")
print(f"Total SRAM available:      {total_sram_mb:.1f} MB")
print(f"Model is {llama_1b_bytes / 1e6 / total_sram_mb:.0f}× larger than total SRAM.")
print()
print("SRAM can hold small tiles of data for computation,")
print("but the model weights MUST live in HBM.")
print("The goal of fusion: avoid unnecessary HBM traffic for ACTIVATIONS.")

## 3 — Triton Hello World: Vector Add

Triton is a language for writing GPU kernels in Python.
Key concepts:

- **`program_id`** — which block of work this thread-block handles
- **Block pointers** — each program processes a contiguous block of elements
- **Masking** — handle arrays that don't divide evenly by block size
- **`tl.load` / `tl.store`** — read/write from/to HBM

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def vector_add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # YOUR CODE
    # Hint:
    # 1. Get program ID:  pid = tl.program_id(0)
    # 2. Compute offsets:  offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    # 3. Create mask:      mask = offsets < n_elements
    # 4. Load inputs:      x = tl.load(x_ptr + offsets, mask=mask)
    #                      y = tl.load(y_ptr + offsets, mask=mask)
    # 5. Compute:          out = x + y
    # 6. Store result:     tl.store(out_ptr + offsets, out, mask=mask)
    pass

def vector_add_triton(x, y):
    assert x.shape == y.shape
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK_SIZE = 1024
    grid = ((n + BLOCK_SIZE - 1) // BLOCK_SIZE,)
    vector_add_kernel[grid](x, y, out, n, BLOCK_SIZE=BLOCK_SIZE)
    return out

# Test correctness
n = 100_000
x = torch.randn(n, device='cuda', dtype=torch.float16)
y = torch.randn(n, device='cuda', dtype=torch.float16)

out_triton = vector_add_triton(x, y)
out_torch = x + y
print(f"Max diff: {(out_triton - out_torch).abs().max().item():.2e}")

# Benchmark vs PyTorch
n_large = 10_000_000
x = torch.randn(n_large, device='cuda', dtype=torch.float16)
y = torch.randn(n_large, device='cuda', dtype=torch.float16)

# Warmup
for _ in range(10):
    _ = vector_add_triton(x, y)
    _ = x + y

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(100):
    _ = vector_add_triton(x, y)
end.record()
torch.cuda.synchronize()
triton_ms = start.elapsed_time(end) / 100

start.record()
for _ in range(100):
    _ = x + y
end.record()
torch.cuda.synchronize()
torch_ms = start.elapsed_time(end) / 100

print(f"\nTriton: {triton_ms:.3f} ms")
print(f"PyTorch: {torch_ms:.3f} ms")
print(f"Ratio: {triton_ms / torch_ms:.2f}× (should be ~1.0 — vector add is trivial)")

## 4 — Tiling and Memory Access

### Why tiling improves bandwidth utilization

Without tiling: each element is loaded from HBM, used once, discarded.
With tiling: a block of elements is loaded into SRAM, used for ALL
operations in the fused kernel, then the result is written back.

The **data reuse** within a tile is what reduces HBM traffic.
Larger tiles = more reuse, but limited by SRAM capacity.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def fused_add_relu_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    """Fused: out = relu(x + y) — one read of x, y; one write of out."""
    # YOUR CODE
    # Hint: same pattern as vector_add, but add a relu step:
    # 1. pid = tl.program_id(0)
    # 2. offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    # 3. mask = offsets < n_elements
    # 4. x = tl.load(x_ptr + offsets, mask=mask)
    #    y = tl.load(y_ptr + offsets, mask=mask)
    # 5. out = tl.maximum(x + y, 0.0)  # fused add + relu
    # 6. tl.store(out_ptr + offsets, out, mask=mask)
    pass

def fused_add_relu(x, y):
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK_SIZE = 1024
    grid = ((n + BLOCK_SIZE - 1) // BLOCK_SIZE,)
    fused_add_relu_kernel[grid](x, y, out, n, BLOCK_SIZE=BLOCK_SIZE)
    return out

# Compare: fused vs unfused (2 separate ops)
n = 10_000_000
x = torch.randn(n, device='cuda', dtype=torch.float16)
y = torch.randn(n, device='cuda', dtype=torch.float16)

# Correctness
ref = torch.relu(x + y)
fused = fused_add_relu(x, y)
print(f"Max diff: {(ref - fused).abs().max().item():.2e}")

# Bandwidth analysis
tensor_bytes = n * 2  # FP16
unfused_hbm = 3 * tensor_bytes + 3 * tensor_bytes  # add: read x,y write tmp; relu: read tmp write out
fused_hbm = 2 * tensor_bytes + tensor_bytes         # read x,y, write out
print(f"\nUnfused HBM traffic: {unfused_hbm / 1e6:.1f} MB")
print(f"Fused HBM traffic:   {fused_hbm / 1e6:.1f} MB")
print(f"Bandwidth saved:     {1 - fused_hbm / unfused_hbm:.0%}")

## 5 — Fused RMSNorm Kernel

### The money kernel

RMSNorm is used before every attention and FFN block in LLaMA.
That's 2 × 16 = 32 RMSNorm calls per forward pass.

**Unfused** (standard PyTorch):
1. Read x from HBM, compute x² mean, write intermediate
2. Read x and intermediate, normalize, write x_norm
→ **2 roundtrips** through HBM

**Fused** (Triton kernel):
1. Read x from HBM once
2. Compute x² mean in registers/SRAM
3. Normalize in registers/SRAM
4. Write x_norm to HBM once
→ **1 read, 1 write**

Formula: `rms_norm(x) = x * weight / sqrt(mean(x²) + eps)`

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def rms_norm_kernel(
    x_ptr, weight_ptr, out_ptr,
    n_rows, n_cols,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    """Fused RMSNorm: one read, one write per row."""
    row_idx = tl.program_id(0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    mask = col_offsets < n_cols

    # YOUR CODE
    # Hint:
    # 1. Compute row pointer:  row_start = row_idx * n_cols
    # 2. Load row:             x = tl.load(x_ptr + row_start + col_offsets, mask=mask, other=0.0)
    # 3. Compute RMS:          x_sq = x * x
    #                          mean_sq = tl.sum(x_sq, axis=0) / n_cols
    #                          rms = tl.rsqrt(mean_sq + eps)  # 1/sqrt(mean(x^2) + eps)
    # 4. Load weight:          w = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
    # 5. Normalize:            out = x * rms * w
    # 6. Store:                tl.store(out_ptr + row_start + col_offsets, out, mask=mask)
    pass

def rms_norm_triton(x, weight, eps=1e-6):
    n_rows, n_cols = x.shape
    out = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    grid = (n_rows,)
    rms_norm_kernel[grid](x, weight, out, n_rows, n_cols, eps, BLOCK_SIZE=BLOCK_SIZE)
    return out

# Test correctness
def rms_norm_pytorch(x, weight, eps=1e-6):
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    return x / rms * weight

rows, cols = 512, 2048  # (batch, d_model)
x = torch.randn(rows, cols, device='cuda', dtype=torch.float32)
weight = torch.ones(cols, device='cuda', dtype=torch.float32)

ref = rms_norm_pytorch(x, weight)
triton_out = rms_norm_triton(x, weight)

if triton_out is not None:
    print(f"Max diff: {(ref - triton_out).abs().max().item():.2e}")
else:
    print("Implement rms_norm_kernel above!")

# Benchmark
print("\nBenchmarking RMSNorm:")
for size in [(512, 2048), (2048, 2048), (8192, 2048)]:
    r, c = size
    x = torch.randn(r, c, device='cuda', dtype=torch.float32)
    w = torch.ones(c, device='cuda', dtype=torch.float32)

    # Warmup
    for _ in range(10):
        _ = rms_norm_pytorch(x, w)
        _ = rms_norm_triton(x, w)

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(100):
        _ = rms_norm_pytorch(x, w)
    end.record()
    torch.cuda.synchronize()
    pytorch_us = start.elapsed_time(end) / 100 * 1000

    start.record()
    for _ in range(100):
        _ = rms_norm_triton(x, w)
    end.record()
    torch.cuda.synchronize()
    triton_us = start.elapsed_time(end) / 100 * 1000

    print(f"  ({r:>5d}, {c:>5d}): PyTorch={pytorch_us:.1f} μs, Triton={triton_us:.1f} μs, speedup={pytorch_us/triton_us:.2f}×")

## 6 — Fused SwiGLU Activation

### SwiGLU in LLaMA's FFN

LLaMA uses SwiGLU activation in the feed-forward block:

```python
gate = W_gate @ x      # (d_model → d_ff)
up   = W_up @ x        # (d_model → d_ff)
out  = silu(gate) * up  # elementwise
```

The `silu(gate) * up` part is two elementwise ops that PyTorch
executes as separate kernels with an HBM roundtrip between them.

Fusing: `silu(gate) * up` in one kernel = 1 read (gate, up) + 1 write.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def fused_swiglu_kernel(
    gate_ptr, up_ptr, out_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    """Fused SwiGLU: out = silu(gate) * up in one kernel."""
    # YOUR CODE
    # Hint:
    # 1. pid = tl.program_id(0)
    # 2. offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    # 3. mask = offsets < n_elements
    # 4. gate = tl.load(gate_ptr + offsets, mask=mask)
    #    up   = tl.load(up_ptr + offsets, mask=mask)
    # 5. SiLU(x) = x * sigmoid(x)
    #    silu_gate = gate * tl.sigmoid(gate)
    # 6. out = silu_gate * up
    # 7. tl.store(out_ptr + offsets, out, mask=mask)
    pass

def fused_swiglu(gate, up):
    out = torch.empty_like(gate)
    n = gate.numel()
    BLOCK_SIZE = 1024
    grid = ((n + BLOCK_SIZE - 1) // BLOCK_SIZE,)
    fused_swiglu_kernel[grid](gate, up, out, n, BLOCK_SIZE=BLOCK_SIZE)
    return out

def unfused_swiglu(gate, up):
    return torch.nn.functional.silu(gate) * up

# Test correctness
n = 512 * 8192  # batch * d_ff for LLaMA-1B
gate = torch.randn(n, device='cuda', dtype=torch.float16)
up   = torch.randn(n, device='cuda', dtype=torch.float16)

ref = unfused_swiglu(gate, up)
fused = fused_swiglu(gate, up)

if fused is not None:
    print(f"Max diff: {(ref - fused).abs().max().item():.2e}")
else:
    print("Implement fused_swiglu_kernel above!")

# Benchmark
for _ in range(10):
    _ = unfused_swiglu(gate, up)
    _ = fused_swiglu(gate, up)

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(100):
    _ = unfused_swiglu(gate, up)
end.record()
torch.cuda.synchronize()
unfused_us = start.elapsed_time(end) / 100 * 1000

start.record()
for _ in range(100):
    _ = fused_swiglu(gate, up)
end.record()
torch.cuda.synchronize()
fused_us = start.elapsed_time(end) / 100 * 1000

print(f"\nUnfused: {unfused_us:.1f} μs")
print(f"Fused:   {fused_us:.1f} μs")
print(f"Speedup: {unfused_us / fused_us:.2f}×")

# HBM traffic analysis
elem_bytes = 2  # FP16
unfused_traffic = n * elem_bytes * 5  # read gate, write silu_gate, read silu_gate, read up, write out
fused_traffic   = n * elem_bytes * 3  # read gate, read up, write out
print(f"\nUnfused HBM: {unfused_traffic / 1e6:.1f} MB")
print(f"Fused HBM:   {fused_traffic / 1e6:.1f} MB")
print(f"Saved:       {(unfused_traffic - fused_traffic) / 1e6:.1f} MB ({1 - fused_traffic/unfused_traffic:.0%})")

## 7 — Tiled Matrix Multiply

### The core of deep learning: matmul

Matrix multiply is the most important operation in transformers.
cuBLAS/cuDNN are extremely optimized for this, but understanding the
tiled algorithm is essential for kernel development.

Key idea: break matrices into blocks that fit in SRAM, multiply
block-by-block, accumulate results.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    """Simplified tiled matmul: C[M,N] = A[M,K] @ B[K,N]."""
    # YOUR CODE
    # Hint:
    # 1. Get block indices:
    #    pid_m = tl.program_id(0)
    #    pid_n = tl.program_id(1)
    #
    # 2. Compute row/col offsets for this block:
    #    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)  # rows of C
    #    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)  # cols of C
    #
    # 3. Initialize accumulator:
    #    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    #
    # 4. Loop over K dimension in blocks:
    #    for k in range(0, K, BLOCK_K):
    #        rk = k + tl.arange(0, BLOCK_K)
    #        # Load A block [BLOCK_M, BLOCK_K]
    #        a = tl.load(a_ptr + rm[:, None]*stride_am + rk[None, :]*stride_ak,
    #                    mask=(rm[:, None] < M) & (rk[None, :] < K), other=0.0)
    #        # Load B block [BLOCK_K, BLOCK_N]
    #        b = tl.load(b_ptr + rk[:, None]*stride_bk + rn[None, :]*stride_bn,
    #                    mask=(rk[:, None] < K) & (rn[None, :] < N), other=0.0)
    #        acc += tl.dot(a, b)
    #
    # 5. Store result:
    #    tl.store(c_ptr + rm[:, None]*stride_cm + rn[None, :]*stride_cn,
    #             acc.to(tl.float16),
    #             mask=(rm[:, None] < M) & (rn[None, :] < N))
    pass

def matmul_triton(a, b):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2
    c = torch.empty(M, N, device=a.device, dtype=a.dtype)
    BLOCK_M, BLOCK_N, BLOCK_K = 64, 64, 32
    grid = ((M + BLOCK_M - 1) // BLOCK_M, (N + BLOCK_N - 1) // BLOCK_N)
    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    return c

# Test correctness
M, K, N = 512, 2048, 8192  # LLaMA-1B: (batch, d_model) @ (d_model, d_ff)
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

ref = a @ b
triton_out = matmul_triton(a, b)

if triton_out is not None:
    # FP16 matmul tolerance is larger
    rel_err = ((ref - triton_out).abs() / (ref.abs() + 1e-6)).mean().item()
    print(f"Mean relative error: {rel_err:.2e}")
else:
    print("Implement matmul_kernel above!")

# Benchmark vs cuBLAS
for _ in range(10):
    _ = a @ b
    _ = matmul_triton(a, b)

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(100):
    _ = a @ b
end.record()
torch.cuda.synchronize()
cublas_us = start.elapsed_time(end) / 100 * 1000

start.record()
for _ in range(100):
    _ = matmul_triton(a, b)
end.record()
torch.cuda.synchronize()
triton_us = start.elapsed_time(end) / 100 * 1000

flops = 2 * M * K * N
cublas_tflops = flops / (cublas_us * 1e-6) / 1e12
triton_tflops = flops / (triton_us * 1e-6) / 1e12

print(f"\ncuBLAS:  {cublas_us:.1f} μs  ({cublas_tflops:.1f} TFLOPS)")
print(f"Triton:  {triton_us:.1f} μs  ({triton_tflops:.1f} TFLOPS)")
print(f"Ratio:   {triton_us / cublas_us:.2f}× (cuBLAS is heavily optimized — don't expect to match it)")

## 8 — Performance Analysis

For each kernel we wrote, let's calculate the arithmetic intensity
and see where they sit on the roofline.

Key insight: **fusion moves operations to the RIGHT on the roofline**
(same FLOPs, fewer bytes → higher AI).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A100 parameters
bw_gb_s = 1774
peak_tflops = 237
ridge = peak_tflops * 1e3 / bw_gb_s  # ~134 FLOP/byte

# ── Arithmetic Intensity for each kernel ─────────────────────────────
# Shape: (512, 2048), FP16
B, D = 512, 2048
D_FF = 8192
elem = 2  # FP16 bytes

# YOUR CODE: calculate AI for each operation
# 
# RMSNorm unfused:
#   FLOPs ≈ B * D * 5  (square, sum, sqrt, divide, multiply)
#   Bytes = 4 * B * D * elem  (2 reads + 2 writes of tensor)
#   AI_unfused = FLOPs / Bytes
#
# RMSNorm fused:
#   FLOPs = same
#   Bytes = 2 * B * D * elem  (1 read + 1 write)
#   AI_fused = FLOPs / Bytes
#
# SwiGLU unfused:
#   FLOPs ≈ B * D_FF * 4  (sigmoid, mul, mul — ~4 ops per element)
#   Bytes = 5 * B * D_FF * elem  (read gate, write silu_gate, read silu_gate, read up, write out)
#   AI_unfused = FLOPs / Bytes
#
# SwiGLU fused:
#   FLOPs = same
#   Bytes = 3 * B * D_FF * elem  (read gate, read up, write out)
#   AI_fused = FLOPs / Bytes
#
# Matmul (d_model → d_ff):
#   FLOPs = 2 * B * D * D_FF
#   Bytes = (B * D + D * D_FF + B * D_FF) * elem
#   AI = FLOPs / Bytes

# Example structure (fill in YOUR CODE values):
ops = {
    'RMSNorm\n(unfused)':  {'ai': 1.25, 'label': 'mem-bound'},
    'RMSNorm\n(fused)':    {'ai': 2.5,  'label': 'mem-bound'},
    'SwiGLU\n(unfused)':   {'ai': 0.5,  'label': 'mem-bound'},
    'SwiGLU\n(fused)':     {'ai': 0.67, 'label': 'mem-bound'},
    'Matmul\n(d→d_ff)':    {'ai': 95,   'label': 'near ridge'},
}

# Plot roofline
ai_range = np.logspace(-1, 4, 500)
perf = np.minimum(ai_range * bw_gb_s / 1e3, peak_tflops)

fig, ax = plt.subplots(figsize=(12, 7))
ax.loglog(ai_range, perf, 'k-', linewidth=2, label='A100 Roofline')
ax.axvline(ridge, color='gray', linestyle='--', alpha=0.5, label=f'Ridge: {ridge:.0f} FLOP/byte')

colors = {'unfused': 'red', 'fused': 'green', 'near ridge': 'blue'}
for name, info in ops.items():
    ai = info['ai']
    achieved = min(ai * bw_gb_s / 1e3, peak_tflops)
    color = 'red' if 'unfused' in name else ('green' if 'fused' in name else 'blue')
    marker = 'v' if 'unfused' in name else ('^' if 'fused' in name else 'D')
    ax.plot(ai, achieved, marker, color=color, markersize=12)
    ax.annotate(name, (ai, achieved), textcoords='offset points',
                xytext=(10, 10), fontsize=8, ha='left')

# Draw arrows from unfused to fused
ax.annotate('', xy=(2.5, min(2.5*bw_gb_s/1e3, peak_tflops)),
            xytext=(1.25, min(1.25*bw_gb_s/1e3, peak_tflops)),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
ax.set_ylabel('Performance (TFLOPS)', fontsize=12)
ax.set_title('Kernel Fusion on the A100 Roofline', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Fusion moves elementwise ops RIGHT on the roofline.")
print("They're still memory-bound, but waste less bandwidth.")
print(f"Matmul is already near the ridge point ({ridge:.0f} FLOP/byte) — fusion doesn't help much.")

## 9 — When to Write Custom Kernels vs Use torch.compile

### Practical guidance

`torch.compile` (Chapter 6) handles most fusion automatically:
- Elementwise fusion (RMSNorm, activations, residuals)
- Simple reductions
- Pointwise chains

**Write custom Triton kernels when you need:**

1. **Custom memory layouts** — e.g., paged KV cache (vLLM), ragged
   tensors for variable-length batching
2. **Novel algorithms** — e.g., Flash Attention (tiling + online
   softmax is too complex for torch.compile to discover)
3. **Peak performance** — when torch.compile's output isn't fast enough
   and you need hand-tuned block sizes, prefetching, etc.
4. **Quantized operations** — mixed-precision matmuls (FP8, INT4)
   with custom dequantization

**The 80/20 rule:** `torch.compile` gets you 80% of the way.
Custom kernels are for the last 20% where it matters most.

| Approach | Effort | Typical Speedup | Best For |
|----------|--------|----------------|----------|
| `torch.compile` | Minutes | 1.5-3× | Most workloads |
| Triton kernels | Hours-days | 2-5× | Memory-bound ops |
| CUDA kernels | Days-weeks | 2-10× | Maximum performance |

## Revision Notes

| Topic | Key Insight | Formula / Number |
|-------|-------------|------------------|
|       |             |                  |
|       |             |                  |
|       |             |                  |
|       |             |                  |
|       |             |                  |